<a href="https://colab.research.google.com/github/monicachet99/CSS/blob/main/Q%26A_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kaggle sentence-transformers faiss-cpu transformers pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 63.7 MB/s eta 0:00:00


In [6]:
import os
import pandas as pd
import fitz # pymupdf

!pip install kaggle -qq

dataset_identifier = "piyushsingh80768/constitution-of-india"

print(f"Downloading dataset: {dataset_identifier}...")
!kaggle datasets download -d {dataset_identifier} -q

downloaded_zip_file = dataset_identifier.split('/')[-1] + '.zip'
print(f"Downloaded file: {downloaded_zip_file}")

print(f"Unzipping {downloaded_zip_file}...")
!unzip -o {downloaded_zip_file} -d .

print("\nFiles extracted:")
!ls

file_to_load = '20240716890312078.pdf'

try:
    # Use pymupdf to open and extract text from the PDF
    doc = fitz.open(file_to_load)
    text_content = ""
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        text_content += page.get_text()
    doc.close()

    # Optionally, you can store the text content in a pandas DataFrame or process it directly
    # For now, let's just print a snippet to confirm it worked
    print(f"\nSuccessfully extracted text from '{file_to_load}'.")
    print("First 500 characters of the extracted text:")
    print(text_content[:500])
    print(f"\nTotal length of extracted text: {len(text_content)} characters")

except FileNotFoundError:
    print(f"\nError: The file '{file_to_load}' was not found after unzipping.")
    print("Please check the 'Files extracted:' output above and adjust 'file_to_load' variable if necessary.")
except Exception as e:
    print(f"\nAn error occurred while processing the PDF: {e}")

Dataset URL: https://www.kaggle.com/datasets/piyushsingh80768/constitution-of-india
License(s): CC0-1.0
Downloaded file: constitution-of-india.zip
Unzipping constitution-of-india.zip...
Archive:  constitution-of-india.zip
  inflating: ./20240716890312078.pdf  

Files extracted:
20240716890312078.pdf  constitution-of-india.zip  sample_data

Successfully extracted text from '20240716890312078.pdf'.
First 500 characters of the extracted text:
£ÉÉ®iÉ BÉEÉ ºÉÆÉÊ´ÉvÉÉxÉ 
[1
, 2024
]
THE CONSTITUTION OF INDIA
[As on 1st May, 2024] 
2024
GOVERNMENT OF INDIA
MINISTRY OF LAW AND JUSTICE
LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING
PREFACE
This is the  sixth pocket size edition of the Constitution of 
India in the diglot form. In this edition, the text of the 
Constitution of India has been brought up-to-date by 
incorporating therein all the amendments up to the Constitution 
(One Hundred and Sixth Amendment) Act, 2023. The foot notes 
b

Total length of extracted text: 869171 characters


In [7]:
import re

clean_text = re.sub(r'[^\x00-\x7F]+', ' ', text_content)
clean_text = re.sub(r'\s+', ' ', clean_text).strip()

print("Clean length:", len(clean_text))
print(clean_text[:500])

Clean length: 853451
i B E v x [1 , 2024 ] THE CONSTITUTION OF INDIA [As on 1st May, 2024] 2024 GOVERNMENT OF INDIA MINISTRY OF LAW AND JUSTICE LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING PREFACE This is the sixth pocket size edition of the Constitution of India in the diglot form. In this edition, the text of the Constitution of India has been brought up-to-date by incorporating therein all the amendments up to the Constitution (One Hundred and Sixth Amendment) Act, 2023. The foot notes below the text indicate 


In [8]:
chunks = []

for i in range(0, len(clean_text), 500):
    chunks.append(clean_text[i:i+500])

print("Total chunks:", len(chunks))

Total chunks: 1707


In [10]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Done!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Done!


In [11]:
embeddings = embed_model.encode(chunks, show_progress_bar=True)
print("Shape:", embeddings.shape)

Batches:   0%|          | 0/54 [00:00<?, ?it/s]

Shape: (1707, 384)


In [12]:
import faiss
import numpy as np

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

print("Total vectors:", index.ntotal)

Total vectors: 1707


In [13]:
from transformers import pipeline

qa = pipeline("question-answering", model="deepset/roberta-base-squad2")
print("QA model ready!")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

QA model ready!


In [14]:
def search(question):
    vec = embed_model.encode([question])
    _, ids = index.search(np.array(vec), 5)
    return [chunks[i] for i in ids[0]]

In [15]:
def ask(question):
    results = search(question)
    answers = []

    for chunk in results:
        out = qa(question=question, context=chunk)
        answers.append(out)

    best = max(answers, key=lambda x: x["score"])
    print("Question:", question)
    print("Answer  :", best["answer"])
    print("Score   :", f"{best['score']:.0%}")
    print("-" * 50)

In [16]:
ask("What is Article 21?")
ask("What are fundamental rights?")
ask("How is the President elected?")

Question: What is Article 21?
Answer  : means of production to the common detriment
Score   : 0%
--------------------------------------------------
Question: What are fundamental rights?
Answer  : 19 Right to Constitutional Remedies
Score   : 5%
--------------------------------------------------
Question: How is the President elected?
Answer  : as a member of the House of the People
Score   : 1%
--------------------------------------------------


In [ ]:
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# --- STEP 1: CLEAN TEXT ---
clean_text = re.sub(r'[^\x00-\x7F]+', ' ', text_content)
clean_text = re.sub(r'\s+', ' ', clean_text).strip()
print("✅ Text cleaned!")

# --- STEP 2: CHUNK TEXT ---
chunks = []
paragraphs = clean_text.split('. ')
current_chunk = ""

for para in paragraphs:
    current_chunk += para + ". "
    if len(current_chunk) > 1000:
        chunks.append(current_chunk.strip())
        current_chunk = ""

if current_chunk:
    chunks.append(current_chunk.strip())

print(f"✅ Total chunks: {len(chunks)}")

# --- STEP 3: LOAD EMBEDDING MODEL ---
print("⏳ Loading embedding model...")
embed_model = SentenceTransformer("all-mpnet-base-v2")
print("✅ Embedding model loaded!")

# --- STEP 4: CREATE VECTORS ---
print("⏳ Creating vectors...")
embeddings = embed_model.encode(chunks, show_progress_bar=True)
print(f"✅ Vectors shape: {embeddings.shape}")

# --- STEP 5: BUILD SEARCH INDEX ---
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))
print(f"✅ Search index ready! Total vectors: {index.ntotal}")

# --- STEP 6: LOAD QA MODEL ---
print("⏳ Loading QA model...")
qa = pipeline("question-answering", model="deepset/deberta-v3-base-squad2")
print("✅ QA model ready!")

# --- STEP 7: SEARCH FUNCTION ---
def search(question):
    vec = embed_model.encode([question])
    _, ids = index.search(np.array(vec), 10)
    return [chunks[i] for i in ids[0]]

# --- STEP 8: ASK FUNCTION ---
def ask(question):
    results = search(question)
    answers = []
    for chunk in results:
        out = qa(question=question, context=chunk)
        answers.append(out)
    answers.sort(key=lambda x: x["score"], reverse=True)
    best = answers[0]
    print("Question :", question)
    print("Answer   :", best["answer"])
    print("Score    :", f"{best['score']:.0%}")
    print("-" * 50)

print("\n🎉 ALL DONE! Ready to answer questions!\n")

# --- STEP 9: TEST ---
ask("What is Article 21?")
ask("What are fundamental rights?")
ask("How is the President elected?")

✅ Text cleaned!
✅ Total chunks: 691
⏳ Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!
⏳ Creating vectors...


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [ ]:
while True:
    q = input("Ask: ")
    if q == "exit":
        break
    ask(q)